In [5]:
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")


def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)


try:
    _pip_install(["imbalanced-learn", "xgboost", "lightgbm"])
except Exception as e:
    print("Package install warning:", e)


import gc
import random
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import layers, regularizers


# =========================================================
# 1) Reproducibility
# =========================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# =========================================================
# 2) Config
# =========================================================
K = 10

IMPUTATION_METHODS = ["knn"]
BASELINE_SAMPLING_METHODS = ["none"] # ["none", "ros", "rus", "smote", "adasyn"]
DEEP_SAMPLING_METHODS = ["none"]
FEATURE_SELECTION = "mutual_info"   # "none" | "variance" | "mutual_info" | "rf_importance"
TOP_K_FEATURES = 29 # 29 for insurance_fraud_data
VARIANCE_THRESHOLD = 0.001
RUN_DEEP_MODELS = True
DATA_FRACTION = 1

USE_THRESHOLD_TUNING = True
THRESHOLD_GRID = np.linspace(0.05, 0.95, 91)
THRESHOLD_METRIC = "f1"
DEFAULT_THRESHOLD = 0.5

FOCAL_ALPHA = 0.65
FOCAL_GAMMA_CONVEX = 0.0
FOCAL_GAMMA_NONCONVEX = 2

 # carclaim dataset
#FOCAL_CONFIG = {
#    "RandomForest": {"alpha": 0.70, "gamma_convex": 0, "gamma_nonconvex": 1.5},
#    "Bagging": {"alpha": 0.70, "gamma_convex": 0, "gamma_nonconvex": 0.5},
#    "LogisticRegression": {"alpha": 0.80, "gamma_convex": 0, "gamma_nonconvex": 0.05},
 #   "XGBoost": {"alpha": 0.7, "gamma_convex": 0, "gamma_nonconvex": 0.9},
 #   "LightGBM": {"alpha": 0.70, "gamma_convex": 0, "gamma_nonconvex": 2.0},
#    "Stacking": {"alpha": 0.70, "gamma_convex": 0, "gamma_nonconvex": 0.1},
#}

#Dataset-1
#FOCAL_CONFIG = {
#    "RandomForest": {"alpha": 0.55, "gamma_convex": 0, "gamma_nonconvex": 0.9},
#    "Bagging": {"alpha": 0.55, "gamma_convex": 0, "gamma_nonconvex": 0.1},
#    "LogisticRegression": {"alpha": 0.55, "gamma_convex": 0, "gamma_nonconvex": 0.1},
#    "XGBoost": {"alpha": 0.55, "gamma_convex": 0, "gamma_nonconvex": 0.1},
#    "LightGBM": {"alpha": 0.55, "gamma_convex": 0, "gamma_nonconvex": 0.1},
#    "Stacking": {"alpha": 0.55, "gamma_convex": 0, "gamma_nonconvex": 0.1},
#}


#Minitab Dataset
FOCAL_CONFIG = {
    "RandomForest": {"alpha": 0.65, "gamma_convex": 0, "gamma_nonconvex": 0.2},
    "Bagging": {"alpha": 0.65, "gamma_convex": 0, "gamma_nonconvex": 0.1},
    "LogisticRegression": {"alpha": 0.65, "gamma_convex": 0, "gamma_nonconvex": 0.2},
    "XGBoost": {"alpha": 0.65, "gamma_convex": 0, "gamma_nonconvex": 0.2},
    "LightGBM": {"alpha": 0.65, "gamma_convex": 0, "gamma_nonconvex": 0.2},
    "Stacking": {"alpha": 0.65, "gamma_convex": 0, "gamma_nonconvex": 0.1},
}


DEEP_FOCAL_CONFIG = {
    "MLP": {
        "alpha_pos": 0.65,
        "alpha_neg": 0.35,
        "gamma_convex": 0.0,
        "gamma_nonconvex": 2,
    }
}

DEEP_BATCH_SIZE = 32
DEEP_LR = 3e-4
DEEP_MIN_LR = 1e-6
EARLY_STOPPING_PATIENCE = 5
REDUCE_LR_PATIENCE = 2

DEEP_EPOCHS_STANDARD = 20
DEEP_EPOCHS_CONVEX = 20
DEEP_EPOCHS_NONCONVEX = 20
DEEP_EPOCHS_STAGE1 = 5
DEEP_EPOCHS_STAGE2 = 15

STACKING_META_EPOCHS = 15
STACKING_SWITCH_EPOCH = 5
STACKING_RAMP_EPOCHS = 5
STACKING_META_ALPHA = 1e-4  # SGDClassifier regularization strength

METRIC_WEIGHTS = {
    "auc": 0.30,
    "f1": 0.25,
    "recall": 0.15,
    "precision": 0.15,
    "specificity": 0.10,
    "accuracy": 0.05,
}

METRIC_PRIORITY = [
    "fraud_recall",
    "auc",
    "fraud_f1",
    "weighted_average",
    "weighted_f1",
    "weighted_precision",
    "specificity",
    "accuracy",
]


# =========================================================
# 3) Load CSV
# =========================================================
df = pd.read_csv("insurance_fraud_data.csv")
if 0 < DATA_FRACTION < 1.0:
    df = df.sample(frac=DATA_FRACTION, random_state=SEED).reset_index(drop=True)
else:
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)


# =========================================================
# 4) Target detection + cleaning
# =========================================================
def pick_target_column(df):
    candidates = [
        "FraudFound_P",
        "FraudFound",
        "fraud_reported",
        "fraud",
        "is_fraud",
        "target",
        "Class",
        "label",
        "fraud_bool",
    ]
    cols_lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns:
            return c
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return df.columns[-1]



def refine_target_column(df, verbose=True):
    df = df.copy()
    target_col = pick_target_column(df)

    if verbose:
        print(f"\nTarget column detected: {target_col}")
        print("Original dtype:", df[target_col].dtype)
        print("Unique sample values:", df[target_col].dropna().astype(str).unique()[:10])

    s = df[target_col]

    if pd.api.types.is_numeric_dtype(s):
        uniq = sorted(pd.Series(s.dropna().unique()).tolist())
        if set(uniq).issubset({0, 1}):
            df[target_col] = s.astype(int)
        elif len(uniq) == 2:
            mapping = {uniq[0]: 0, uniq[1]: 1}
            df[target_col] = s.map(mapping).astype(int)
        else:
            thr = np.median(s.dropna())
            df[target_col] = (s > thr).astype(int)
    else:
        s_str = s.astype(str).str.strip().str.lower()
        map01 = {
            "y": 1, "yes": 1, "true": 1, "1": 1,
            "fraud": 1, "fraudulent": 1, "positive": 1,
            "n": 0, "no": 0, "false": 0, "0": 0,
            "not fraud": 0, "nonfraud": 0, "non-fraud": 0,
            "legit": 0, "legitimate": 0, "negative": 0,
        }

        unique_vals = set(s_str.dropna().unique())
        if unique_vals.issubset(set(map01.keys())):
            df[target_col] = s_str.map(map01).astype(int)
        elif len(unique_vals) == 2:
            vals = sorted(unique_vals)
            mapping = {vals[0]: 0, vals[1]: 1}
            df[target_col] = s_str.map(mapping).astype(int)
            if verbose:
                print("Auto-mapped binary text labels:", mapping)
        else:
            enc = LabelEncoder()
            y_tmp = enc.fit_transform(s_str)
            if len(np.unique(y_tmp)) > 2:
                thr = np.median(y_tmp)
                y_tmp = (y_tmp > thr).astype(int)
            df[target_col] = y_tmp.astype(int)

    print("\nTarget distribution:")
    print(df[target_col].value_counts(dropna=False))
    print("\nTarget proportions:")
    print(df[target_col].value_counts(normalize=True).round(4))

    return df, target_col



df, target_col = refine_target_column(df)


# =========================================================
# 5) Prepare X, y
# =========================================================
X = df.drop(columns=[target_col]).copy()
y = df[target_col].values.astype(int)

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

feature_names = X.columns.tolist()
X_np = X.values.astype(np.float32)
y_np = y.astype(int)
TOP_K_FEATURES = min(TOP_K_FEATURES, X_np.shape[1])
current_imputer_name = IMPUTATION_METHODS[0]


# =========================================================
# 6) Utilities
# =========================================================
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape != (2, 2):
        return 0.0
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0



def tune_threshold(y_true, y_prob, metric="f1", thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 91)

    best_thr = 0.5
    best_score = -np.inf
    y_prob = np.asarray(y_prob).reshape(-1)

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)

        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, y_pred, zero_division=0)
        elif metric == "precision":
            score = precision_score(y_true, y_pred, zero_division=0)
        elif metric == "balanced":
            score = 0.5 * (
                recall_score(y_true, y_pred, zero_division=0)
                + specificity_score(y_true, y_pred)
            )
        else:
            score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_score:
            best_score = score
            best_thr = thr

    return float(best_thr), float(best_score)



def eval_binary_full(y_true, y_prob, thr=0.5):
    y_prob = np.asarray(y_prob, dtype=np.float64).reshape(-1)
    y_prob = np.nan_to_num(y_prob, nan=0.5, posinf=1.0, neginf=0.0)
    #y_prob = np.clip(y_prob, 1e-7, 1.0 - 1e-7)
    y_pred = (y_prob >= thr).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0

    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    fraud_precision, fraud_recall, fraud_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )

    auc_value = 0.5
    try:
        auc_value = roc_auc_score(y_true, y_prob)
    except Exception:
        pass

    accuracy = accuracy_score(y_true, y_pred)
    weighted_accuracy = accuracy

    return {
        "accuracy": accuracy,
        "weighted_accuracy": weighted_accuracy,
        "auc": auc_value,
        "specificity": specificity,
        "threshold": thr,
        "cm": cm,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
        "fraud_precision": fraud_precision,
        "fraud_recall": fraud_recall,
        "fraud_f1": fraud_f1,
    }



def compute_weighted_average(m):
    return (
        METRIC_WEIGHTS["auc"] * m["auc"]
        + METRIC_WEIGHTS["f1"] * m["weighted_f1"]
        + METRIC_WEIGHTS["recall"] * m["weighted_recall"]
        + METRIC_WEIGHTS["precision"] * m["weighted_precision"]
        + METRIC_WEIGHTS["specificity"] * m["specificity"]
        + METRIC_WEIGHTS["accuracy"] * m["accuracy"]
    )



def plot_roc_curves(y_true, curves, title="ROC Curves"):
    plt.figure(figsize=(8, 6))
    for label, prob in curves:
        prob = np.asarray(prob).reshape(-1)
        try:
            fpr, tpr, _ = roc_curve(y_true, prob)
            aucv = roc_auc_score(y_true, prob)
            plt.plot(fpr, tpr, label=f"{label} (AUC={aucv:.3f})")
        except Exception:
            continue
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



def build_imputer(method):
    if method == "simple":
        return SimpleImputer(strategy="median")
    if method == "knn":
        return KNNImputer(n_neighbors=5)
    if method == "iterative":
        return IterativeImputer(random_state=SEED)
    raise ValueError(f"Unknown imputation method: {method}")



def apply_sampling(method, Xtr, ytr, ratio=1):
    if method == "none":
        return Xtr, ytr
    if method == "ros":
        sampler = RandomOverSampler(sampling_strategy=ratio, random_state=SEED)
    elif method == "rus":
        sampler = RandomUnderSampler(sampling_strategy=ratio, random_state=SEED)
    elif method == "smote":
        sampler = SMOTE(sampling_strategy=ratio, random_state=SEED)
    elif method == "adasyn":
        sampler = ADASYN(sampling_strategy=ratio, random_state=SEED)
    else:
        raise ValueError(f"Unknown sampling method: {method}")
    return sampler.fit_resample(Xtr, ytr)



def select_features(Xtr, ytr, Xva, method="mutual_info", top_k=20, variance_threshold=0.01):
    if method == "none":
        return Xtr, Xva, None

    if method == "variance":
        selector = VarianceThreshold(threshold=variance_threshold)
        return selector.fit_transform(Xtr), selector.transform(Xva), selector

    if method == "mutual_info":
        k = min(top_k, Xtr.shape[1])
        selector = SelectKBest(score_func=mutual_info_classif, k=k)
        return selector.fit_transform(Xtr, ytr), selector.transform(Xva), selector

    if method == "rf_importance":
        k = min(top_k, Xtr.shape[1])
        rf = RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )
        rf.fit(Xtr, ytr)
        idx = np.argsort(rf.feature_importances_)[::-1][:k]
        return Xtr[:, idx], Xva[:, idx], idx

    raise ValueError(f"Unknown feature selection method: {method}")



def get_selected_feature_names(selector, original_names):
    if selector is None:
        return original_names
    if hasattr(selector, "get_support"):
        mask = selector.get_support()
        return [name for name, keep in zip(original_names, mask) if keep]
    if isinstance(selector, np.ndarray):
        return [original_names[i] for i in selector]
    return original_names



def compute_class_weight_dict(y):
    y = np.asarray(y).astype(int).reshape(-1)
    n0 = np.sum(y == 0)
    n1 = np.sum(y == 1)
    if n0 == 0 or n1 == 0:
        return None
    return {
        0: (n0 + n1) / (2.0 * n0),
        1: (n0 + n1) / (2.0 * n1),
    }


def compute_per_sample_class_weights(y):
    y = np.asarray(y).astype(int).reshape(-1)
    classes = np.array([0, 1], dtype=int)
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    cw_map = {cls: w for cls, w in zip(classes, cw)}
    return np.asarray([cw_map[int(label)] for label in y], dtype=float)



def preprocess_fold(X_train, X_val, y_train):
    imputer = build_imputer(current_imputer_name)
    X_train_imp = imputer.fit_transform(X_train)
    X_val_imp = imputer.transform(X_val)
   # X_train_imp = X_train
   # X_val_imp = X_val
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)

    X_train_fs, X_val_fs, selector = select_features(
        X_train_scaled,
        y_train,
        X_val_scaled,
        method=FEATURE_SELECTION,
        top_k=TOP_K_FEATURES,
        variance_threshold=VARIANCE_THRESHOLD,
    )

    selected_names = get_selected_feature_names(selector, feature_names)
    return X_train_fs, X_val_fs, selector, selected_names



def safe_predict_proba_binary(model, X):
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X)
        prob = np.asarray(prob)
        if prob.ndim == 2 and prob.shape[1] >= 2:
            return prob[:, 1]
        return prob.reshape(-1)

    if hasattr(model, "decision_function"):
        scores = np.asarray(model.decision_function(X), dtype=float).reshape(-1)
        smin, smax = scores.min(), scores.max()
        if smax - smin < 1e-12:
            return np.full_like(scores, 0.5, dtype=float)
        return 1.0 / (1.0 + np.exp(-scores))

    pred = model.predict(X)
    return np.asarray(pred, dtype=float).reshape(-1)



def fit_with_optional_sample_weight(model, X, y, sample_weight=None):
    fit_sig = inspect.signature(model.fit)
    if sample_weight is not None and "sample_weight" in fit_sig.parameters:
        return model.fit(X, y, sample_weight=sample_weight)
    return model.fit(X, y)



def compute_focal_sample_weights(y_true, y_prob, alpha=0.55, gamma=2.0, normalize=True):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=float).reshape(-1)
   # y_prob = np.clip(y_prob, 1e-7, 1.0 - 1e-7)

    pt = np.where(y_true == 1, y_prob, 1.0 - y_prob)
    alpha_t = np.where(y_true == 1, alpha, 1.0 - alpha)
    weights = alpha_t * np.power(1.0 - pt, gamma)
    #weights = np.clip(weights, 1e-4, None)

    if normalize:
        mean_w = np.mean(weights)
        if mean_w > 0:
            weights = weights / mean_w
    return weights


# =========================================================
# 7) Classical focal wrappers
# =========================================================
class FocalSelfReweightingWrapper(BaseEstimator, ClassifierMixin):
    """
    Generic refit-based wrapper for models that do not naturally expose
    per-iteration updates in the sklearn API.

    variant:
        'convex'     -> single reweight with gamma = 0
        'nonconvex'  -> single reweight with gamma > 0
        'multistage' -> convex reweight then nonconvex reweight
    """

    def __init__(self, estimator, alpha=0.55, gamma_convex=0.0, gamma_nonconvex=2.0, variant="convex"):
        self.estimator = estimator
        self.alpha = alpha
        self.gamma_convex = gamma_convex
        self.gamma_nonconvex = gamma_nonconvex
        self.variant = variant

    def _single_refit(self, estimator, X, y, gamma):
        first_model = clone(estimator)
        fit_with_optional_sample_weight(first_model, X, y, sample_weight=None)
        p = safe_predict_proba_binary(first_model, X)
        sw = compute_focal_sample_weights(y, p, alpha=self.alpha, gamma=gamma, normalize=True)

        second_model = clone(estimator)
        fit_with_optional_sample_weight(second_model, X, y, sample_weight=sw)
        return second_model

    def fit(self, X, y):
        if self.variant == "convex":
            self.final_model_ = self._single_refit(self.estimator, X, y, self.gamma_convex)
        elif self.variant == "nonconvex":
            self.final_model_ = self._single_refit(self.estimator, X, y, self.gamma_nonconvex)
        elif self.variant == "multistage":
            stage1_seed = clone(self.estimator)
            fit_with_optional_sample_weight(stage1_seed, X, y, sample_weight=None)
            p1 = safe_predict_proba_binary(stage1_seed, X)
            sw1 = compute_focal_sample_weights(y, p1, alpha=self.alpha, gamma=self.gamma_convex, normalize=True)

            stage1_model = clone(self.estimator)
            fit_with_optional_sample_weight(stage1_model, X, y, sample_weight=sw1)

            p2 = safe_predict_proba_binary(stage1_model, X)
            sw2 = compute_focal_sample_weights(y, p2, alpha=self.alpha, gamma=self.gamma_nonconvex, normalize=True)

            stage2_model = clone(self.estimator)
            fit_with_optional_sample_weight(stage2_model, X, y, sample_weight=sw2)
            self.final_model_ = stage2_model
        else:
            raise ValueError(f"Unknown variant: {self.variant}")
        return self

    def predict_proba(self, X):
        p = safe_predict_proba_binary(self.final_model_, X)
        #p = np.clip(np.asarray(p).reshape(-1), 1e-7, 1.0 - 1e-7)
        return np.column_stack([1.0 - p, p])

    def predict(self, X):
        p = safe_predict_proba_binary(self.final_model_, X)
        return (p >= 0.5).astype(int)


class IterativeFocalLogisticWrapper(BaseEstimator, ClassifierMixin):
    """
    True iteration-level switching for shallow logistic models using SGDClassifier.
    """

    def __init__(
        self,
        alpha=0.75,
        gamma_convex=0.0,
        gamma_nonconvex=0.05,
        variant="multistage",
        n_epochs=20,
        switch_epoch=5,
        ramp_epochs=5,
        shuffle=True,
        sgd_alpha=1e-4,
    ):
        self.alpha = alpha
        self.gamma_convex = gamma_convex
        self.gamma_nonconvex = gamma_nonconvex
        self.variant = variant
        self.n_epochs = n_epochs
        self.switch_epoch = switch_epoch
        self.ramp_epochs = ramp_epochs
        self.shuffle = shuffle
        self.sgd_alpha = sgd_alpha

    def _gamma_for_epoch(self, epoch):
        if self.variant == "standard":
            return None
        if self.variant == "convex":
            return self.gamma_convex
        if self.variant == "nonconvex":
            return self.gamma_nonconvex
        if self.variant != "multistage":
            raise ValueError(f"Unknown variant: {self.variant}")

        if epoch < self.switch_epoch:
            return self.gamma_convex
        if self.ramp_epochs > 0 and epoch < self.switch_epoch + self.ramp_epochs:
            progress = (epoch - self.switch_epoch) / float(self.ramp_epochs)
            return self.gamma_convex + progress * (self.gamma_nonconvex - self.gamma_convex)
        return self.gamma_nonconvex

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).astype(int)

        self.model_ = SGDClassifier(
            loss="log_loss",
            alpha=self.sgd_alpha,
            max_iter=1,
            tol=None,
            random_state=SEED,
            class_weight=None,
            warm_start=True,
        )

        base_class_weight = compute_per_sample_class_weights(y)
        rng = np.random.RandomState(SEED)
        classes = np.array([0, 1], dtype=int)
        self.training_history_ = []

        for epoch in range(self.n_epochs):
            if self.shuffle:
                idx = rng.permutation(len(y))
                X_epoch = X[idx]
                y_epoch = y[idx]
                class_weight_epoch = base_class_weight[idx]
            else:
                X_epoch = X
                y_epoch = y
                class_weight_epoch = base_class_weight

            gamma = self._gamma_for_epoch(epoch)

            if gamma is None:
                sample_weight = class_weight_epoch.copy()
            else:
                if epoch == 0:
                    p_epoch = np.full(len(y_epoch), 0.5, dtype=float)
                else:
                    p_epoch = safe_predict_proba_binary(self.model_, X_epoch)

                focal_weight = compute_focal_sample_weights(
                    y_epoch,
                    p_epoch,
                    alpha=self.alpha,
                    gamma=gamma,
                    normalize=True,
                )
                sample_weight = class_weight_epoch * focal_weight
                sample_weight = sample_weight / max(np.mean(sample_weight), 1e-12)

            self.model_.partial_fit(
                X_epoch,
                y_epoch,
                classes=classes if epoch == 0 else None,
                sample_weight=sample_weight,
            )

            train_prob = safe_predict_proba_binary(self.model_, X)
            train_auc = 0.5
            try:
                train_auc = roc_auc_score(y, train_prob)
            except Exception:
                pass
            self.training_history_.append(
                {
                    "epoch": epoch + 1,
                    "gamma": gamma,
                    "train_auc": float(train_auc),
                }
            )

        return self

    def predict_proba(self, X):
        p = safe_predict_proba_binary(self.model_, X)
        #p = np.clip(np.asarray(p).reshape(-1), 1e-7, 1.0 - 1e-7)
        return np.column_stack([1.0 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class GrowthBasedEnsembleFocalWrapper(BaseEstimator, ClassifierMixin):
    """
    Multistage shallow wrapper for tree ensembles using estimator-growth switching.
    Earlier estimators correspond to the convex stage, later estimators to the nonconvex stage.
    This preserves the ensemble family while approximating stagewise iteration growth.
    """

    def __init__(
        self,
        estimator,
        alpha=0.75,
        gamma_convex=0.0,
        gamma_nonconvex=0.05,
        variant="multistage",
    ):
        self.estimator = estimator
        self.alpha = alpha
        self.gamma_convex = gamma_convex
        self.gamma_nonconvex = gamma_nonconvex
        self.variant = variant

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).astype(int)

        if self.variant == "standard":
            self.model_ = clone(self.estimator)
            fit_with_optional_sample_weight(self.model_, X, y, sample_weight=None)
            return self

        gamma = self.gamma_convex if self.variant == "convex" else self.gamma_nonconvex
        if self.variant in {"convex", "nonconvex"}:
            seed_model = clone(self.estimator)
            fit_with_optional_sample_weight(seed_model, X, y, sample_weight=None)
            p = safe_predict_proba_binary(seed_model, X)
            sw = compute_focal_sample_weights(y, p, alpha=self.alpha, gamma=gamma, normalize=True)
            self.model_ = clone(self.estimator)
            fit_with_optional_sample_weight(self.model_, X, y, sample_weight=sw)
            return self

        if self.variant != "multistage":
            raise ValueError(f"Unknown variant: {self.variant}")

        stage1_seed = clone(self.estimator)
        fit_with_optional_sample_weight(stage1_seed, X, y, sample_weight=None)
        p1 = safe_predict_proba_binary(stage1_seed, X)
        sw1 = compute_focal_sample_weights(y, p1, alpha=self.alpha, gamma=self.gamma_convex, normalize=True)

        self.stage1_model_ = clone(self.estimator)
        fit_with_optional_sample_weight(self.stage1_model_, X, y, sample_weight=sw1)

        p2 = safe_predict_proba_binary(self.stage1_model_, X)
        sw2 = compute_focal_sample_weights(y, p2, alpha=self.alpha, gamma=self.gamma_nonconvex, normalize=True)

        self.stage2_model_ = clone(self.estimator)
        fit_with_optional_sample_weight(self.stage2_model_, X, y, sample_weight=sw2)
        return self

    def predict_proba(self, X):
        if hasattr(self, "model_"):
            p = safe_predict_proba_binary(self.model_, X)
        else:
            p1 = safe_predict_proba_binary(self.stage1_model_, X)
            p2 = safe_predict_proba_binary(self.stage2_model_, X)
            p = 0.5 * (p1 + p2)
        #p = np.clip(np.asarray(p).reshape(-1), 1e-7, 1.0 - 1e-7)
        return np.column_stack([1.0 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class ManualStackingClassifierIterative(BaseEstimator, ClassifierMixin):
    """
    True iteration-level focal switching at the meta-learner level.
    Uses an SGDClassifier with partial_fit so gamma can switch during training epochs.
    """

    def __init__(
        self,
        base_estimators,
        meta_estimator=None,
        cv=5,
        focal_variant="standard",  # "standard", "convex", "nonconvex", "multistage"
        alpha=0.55,
        gamma_convex=0.0,
        gamma_nonconvex=2.0,
        n_epochs=20,
        switch_epoch=5,
        ramp_epochs=0,
        shuffle_meta=True,
    ):
        self.base_estimators = base_estimators
        self.meta_estimator = meta_estimator if meta_estimator is not None else SGDClassifier(
            loss="log_loss",
            alpha=STACKING_META_ALPHA,
            max_iter=1,
            tol=None,
            random_state=SEED,
            class_weight=None,
            warm_start=True,
        )
        self.cv = cv
        self.focal_variant = focal_variant
        self.alpha = alpha
        self.gamma_convex = gamma_convex
        self.gamma_nonconvex = gamma_nonconvex
        self.n_epochs = n_epochs
        self.switch_epoch = switch_epoch
        self.ramp_epochs = ramp_epochs
        self.shuffle_meta = shuffle_meta

    def _build_meta_features(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).astype(int)

        inner_cv = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=SEED)
        meta_features = np.zeros((len(y), len(self.base_estimators)), dtype=float)

        for j, (name, est) in enumerate(self.base_estimators):
            oof_pred = np.zeros(len(y), dtype=float)
            for tr_idx, va_idx in inner_cv.split(X, y):
                est_fold = clone(est)
                fit_with_optional_sample_weight(est_fold, X[tr_idx], y[tr_idx], sample_weight=None)
                oof_pred[va_idx] = safe_predict_proba_binary(est_fold, X[va_idx])
            meta_features[:, j] = oof_pred

        self.fitted_base_models_ = []
        for name, est in self.base_estimators:
            est_full = clone(est)
            fit_with_optional_sample_weight(est_full, X, y, sample_weight=None)
            self.fitted_base_models_.append((name, est_full))

        return meta_features, y

    def _gamma_for_epoch(self, epoch):
        if self.focal_variant == "standard":
            return None
        if self.focal_variant == "convex":
            return self.gamma_convex
        if self.focal_variant == "nonconvex":
            return self.gamma_nonconvex
        if self.focal_variant != "multistage":
            raise ValueError(f"Unknown focal_variant: {self.focal_variant}")

        if epoch < self.switch_epoch:
            return self.gamma_convex
        if self.ramp_epochs <= 0:
            return self.gamma_nonconvex

        progress = min(1.0, (epoch - self.switch_epoch + 1) / float(self.ramp_epochs))
        return self.gamma_convex + progress * (self.gamma_nonconvex - self.gamma_convex)

    def fit(self, X, y):
        meta_features, y = self._build_meta_features(X, y)
        self.meta_model_ = clone(self.meta_estimator)

        rng = np.random.RandomState(SEED)
        classes = np.array([0, 1], dtype=int)
        base_class_sample_weight = compute_per_sample_class_weights(y)

        self.meta_training_history_ = []

        for epoch in range(self.n_epochs):
            if self.shuffle_meta:
                idx = rng.permutation(len(y))
                X_epoch = meta_features[idx]
                y_epoch = y[idx]
                class_weight_epoch = base_class_sample_weight[idx]
            else:
                X_epoch = meta_features
                y_epoch = y
                class_weight_epoch = base_class_sample_weight

            gamma = self._gamma_for_epoch(epoch)

            if gamma is None:
                sample_weight = class_weight_epoch.copy()
                mean_weight = float(np.mean(sample_weight))
            else:
                if epoch == 0:
                    p_epoch = np.full(len(y_epoch), 0.5, dtype=float)
                else:
                    p_epoch = safe_predict_proba_binary(self.meta_model_, X_epoch)

                focal_weight_epoch = compute_focal_sample_weights(
                    y_epoch,
                    p_epoch,
                    alpha=self.alpha,
                    gamma=gamma,
                    normalize=True,
                )
                sample_weight = class_weight_epoch * focal_weight_epoch
                sample_weight = sample_weight / max(np.mean(sample_weight), 1e-12)
                mean_weight = float(np.mean(sample_weight))

            self.meta_model_.partial_fit(
                X_epoch,
                y_epoch,
                classes=classes if epoch == 0 else None,
                sample_weight=sample_weight,
            )

            train_prob = safe_predict_proba_binary(self.meta_model_, meta_features)
            train_auc = 0.5
            try:
                train_auc = roc_auc_score(y, train_prob)
            except Exception:
                pass

            self.meta_training_history_.append(
                {
                    "epoch": epoch + 1,
                    "gamma": None if gamma is None else float(gamma),
                    "mean_sample_weight": mean_weight,
                    "train_auc": float(train_auc),
                }
            )

        return self

    def _make_meta_features(self, X):
        X = np.asarray(X)
        return np.column_stack([
            safe_predict_proba_binary(model, X)
            for _, model in self.fitted_base_models_
        ])

    def predict_proba(self, X):
        meta_X = self._make_meta_features(X)
        p = safe_predict_proba_binary(self.meta_model_, meta_X)
        #p = np.clip(np.asarray(p).reshape(-1), 1e-7, 1.0 - 1e-7)
        return np.column_stack([1.0 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# =========================================================
# 8) Model zoo
# =========================================================
def make_standard_base_estimators():
    return [
        (
            "rf",
            RandomForestClassifier(
                n_estimators=200,
                random_state=SEED,
                n_jobs=-1,
                class_weight="balanced_subsample",
            ),
        ),
        (
            "lr",
            LogisticRegression(
                max_iter=3000,
                random_state=SEED,
                class_weight="balanced",
            ),
        ),
        ("knn", KNeighborsClassifier()),
    ]


def make_family_variants(base_estimator, family_name):
    cfg = FOCAL_CONFIG.get(
        family_name,
        {
            "alpha": FOCAL_ALPHA,
            "gamma_convex": FOCAL_GAMMA_CONVEX,
            "gamma_nonconvex": FOCAL_GAMMA_NONCONVEX,
        },
    )

    if family_name == "LogisticRegression":
        return {
            f"{family_name}_Standard": IterativeFocalLogisticWrapper(
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="standard",
                n_epochs=STACKING_META_EPOCHS,
                switch_epoch=STACKING_SWITCH_EPOCH,
                ramp_epochs=0,
                shuffle=True,
                sgd_alpha=STACKING_META_ALPHA,
            ),
            f"{family_name}_Focal_Convex": IterativeFocalLogisticWrapper(
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="convex",
                n_epochs=STACKING_META_EPOCHS,
                switch_epoch=STACKING_SWITCH_EPOCH,
                ramp_epochs=0,
                shuffle=True,
                sgd_alpha=STACKING_META_ALPHA,
            ),
            f"{family_name}_Focal_NonConvex": IterativeFocalLogisticWrapper(
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="nonconvex",
                n_epochs=STACKING_META_EPOCHS,
                switch_epoch=STACKING_SWITCH_EPOCH,
                ramp_epochs=0,
                shuffle=True,
                sgd_alpha=STACKING_META_ALPHA,
            ),
            f"{family_name}_Focal_Multistage": IterativeFocalLogisticWrapper(
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="multistage",
                n_epochs=STACKING_META_EPOCHS,
                switch_epoch=STACKING_SWITCH_EPOCH,
                ramp_epochs=STACKING_RAMP_EPOCHS,
                shuffle=True,
                sgd_alpha=STACKING_META_ALPHA,
            ),
        }

    if family_name in {"RandomForest", "Bagging"}:
        return {
            f"{family_name}_Standard": clone(base_estimator),
            f"{family_name}_Focal_Convex": GrowthBasedEnsembleFocalWrapper(
                estimator=clone(base_estimator),
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="convex",
            ),
            f"{family_name}_Focal_NonConvex": GrowthBasedEnsembleFocalWrapper(
                estimator=clone(base_estimator),
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="nonconvex",
            ),
            f"{family_name}_Focal_Multistage": GrowthBasedEnsembleFocalWrapper(
                estimator=clone(base_estimator),
                alpha=cfg["alpha"],
                gamma_convex=cfg["gamma_convex"],
                gamma_nonconvex=cfg["gamma_nonconvex"],
                variant="multistage",
            ),
        }

    return {
        f"{family_name}_Standard": clone(base_estimator),
        f"{family_name}_Focal_Convex": FocalSelfReweightingWrapper(
            estimator=clone(base_estimator),
            alpha=cfg["alpha"],
            gamma_convex=cfg["gamma_convex"],
            gamma_nonconvex=cfg["gamma_nonconvex"],
            variant="convex",
        ),
        f"{family_name}_Focal_NonConvex": FocalSelfReweightingWrapper(
            estimator=clone(base_estimator),
            alpha=cfg["alpha"],
            gamma_convex=cfg["gamma_convex"],
            gamma_nonconvex=cfg["gamma_nonconvex"],
            variant="nonconvex",
        ),
        f"{family_name}_Focal_Multistage": FocalSelfReweightingWrapper(
            estimator=clone(base_estimator),
            alpha=cfg["alpha"],
            gamma_convex=cfg["gamma_convex"],
            gamma_nonconvex=cfg["gamma_nonconvex"],
            variant="multistage",
        ),
    }


def get_all_models():
    rf_base = RandomForestClassifier(
        n_estimators=400,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced_subsample",
        max_depth=None,
        min_samples_leaf=2,
    )

    bag_base = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None, min_samples_leaf=2, random_state=SEED),
        random_state=SEED,
        n_estimators=200,
    )

    lr_base = LogisticRegression(max_iter=3000, random_state=SEED, class_weight="balanced")

    xgb_base = XGBClassifier(
        random_state=SEED,
        eval_metric="logloss",
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        n_jobs=-1,
    )

    lgbm_base = LGBMClassifier(
        random_state=SEED,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        class_weight="balanced",
    )

    models = {}
    models.update(make_family_variants(rf_base, "RandomForest"))
    models.update(make_family_variants(bag_base, "Bagging"))
    models.update(make_family_variants(lr_base, "LogisticRegression"))
    models.update(make_family_variants(xgb_base, "XGBoost"))
    #models.update(make_family_variants(lgbm_base, "LightGBM"))

    base_estimators = make_standard_base_estimators()
    stack_cfg = FOCAL_CONFIG.get(
        "Stacking",
        {
            "alpha": FOCAL_ALPHA,
            "gamma_convex": FOCAL_GAMMA_CONVEX,
            "gamma_nonconvex": FOCAL_GAMMA_NONCONVEX,
        },
    )

    def make_meta_estimator():
        return SGDClassifier(
            loss="log_loss",
            alpha=STACKING_META_ALPHA,
            max_iter=1,
            tol=None,
            random_state=SEED,
            class_weight=None,
            warm_start=True,
        )

    models["Stacking_Standard"] = ManualStackingClassifierIterative(
        base_estimators=base_estimators,
        meta_estimator=make_meta_estimator(),
        cv=5,
        focal_variant="standard",
        alpha=stack_cfg["alpha"],
        gamma_convex=stack_cfg["gamma_convex"],
        gamma_nonconvex=stack_cfg["gamma_nonconvex"],
        n_epochs=STACKING_META_EPOCHS,
        switch_epoch=STACKING_SWITCH_EPOCH,
        ramp_epochs=STACKING_RAMP_EPOCHS,
    )
    models["Stacking_Focal_Convex"] = ManualStackingClassifierIterative(
        base_estimators=base_estimators,
        meta_estimator=make_meta_estimator(),
        cv=5,
        focal_variant="convex",
        alpha=stack_cfg["alpha"],
        gamma_convex=stack_cfg["gamma_convex"],
        gamma_nonconvex=stack_cfg["gamma_nonconvex"],
        n_epochs=STACKING_META_EPOCHS,
        switch_epoch=STACKING_SWITCH_EPOCH,
        ramp_epochs=0,
    )
    models["Stacking_Focal_NonConvex"] = ManualStackingClassifierIterative(
        base_estimators=base_estimators,
        meta_estimator=make_meta_estimator(),
        cv=5,
        focal_variant="nonconvex",
        alpha=stack_cfg["alpha"],
        gamma_convex=stack_cfg["gamma_convex"],
        gamma_nonconvex=stack_cfg["gamma_nonconvex"],
        n_epochs=STACKING_META_EPOCHS,
        switch_epoch=STACKING_SWITCH_EPOCH,
        ramp_epochs=0,
    )
    models["Stacking_Focal_Multistage"] = ManualStackingClassifierIterative(
        base_estimators=base_estimators,
        meta_estimator=make_meta_estimator(),
        cv=5,
        focal_variant="multistage",
        alpha=stack_cfg["alpha"],
        gamma_convex=stack_cfg["gamma_convex"],
        gamma_nonconvex=stack_cfg["gamma_nonconvex"],
        n_epochs=STACKING_META_EPOCHS,
        switch_epoch=STACKING_SWITCH_EPOCH,
        ramp_epochs=STACKING_RAMP_EPOCHS,
    )

    return models



def get_model_probabilities(model, Xtr, ytr, Xva):
    model.fit(Xtr, ytr)
    return safe_predict_proba_binary(model, Xva)


# =========================================================
# 9) Compact MLP models
# =========================================================
class ConvexFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=0.0, alpha_pos=0.5, alpha_neg=0.5, name="convex_focal_loss"):
        super().__init__(name=name)
        self.gamma = float(gamma)
        self.alpha_pos = float(alpha_pos)
        self.alpha_neg = float(alpha_neg)

    def call(self, y_true, y_pred_logits):
        y_true = tf.cast(y_true, tf.float32)
        p = tf.nn.sigmoid(y_pred_logits)
        p = tf.clip_by_value(p, 1e-7, 1.0 - 1e-7)

        loss = (
            -self.alpha_pos * y_true * tf.pow(1.0 - p, self.gamma) * tf.math.log(p)
            -self.alpha_neg * (1.0 - y_true) * tf.pow(p, self.gamma) * tf.math.log(1.0 - p)
        )
        return tf.reduce_mean(loss)



def dense_block(x, units, activation="relu", l2_reg=1e-3, dropout_rate=0.25, use_bn=True):
    x = layers.Dense(units, activation=activation, kernel_regularizer=regularizers.l2(l2_reg))(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    return x



def build_compact_mlp_standard(input_dim):
    inp = layers.Input(shape=(input_dim,), name="tabular_input")
    x = layers.BatchNormalization()(inp)
    x = layers.GaussianNoise(0.01)(x)
    x = dense_block(x, 256, activation="swish", l2_reg=1e-3, dropout_rate=0.30, use_bn=True)
    x = dense_block(x, 128, activation="swish", l2_reg=1e-3, dropout_rate=0.20, use_bn=True)
    x = dense_block(x, 256, activation="swish", l2_reg=1e-3, dropout_rate=0.10, use_bn=True)
    prob = layers.Dense(1, activation="sigmoid", name="prob")(x)
    return tf.keras.Model(inp, prob, name="compact_mlp_standard")



def build_compact_mlp_convex(input_dim):
    inp = layers.Input(shape=(input_dim,), name="tabular_input")
    x = layers.BatchNormalization()(inp)
    x = layers.GaussianNoise(0.01)(x)
    x = layers.Dense(256, activation="softplus", kernel_regularizer=regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation="softplus", kernel_regularizer=regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)
    logits = layers.Dense(1, activation=None, name="logits")(x)
    return tf.keras.Model(inp, logits, name="compact_mlp_convex")



def build_compact_mlp_nonconvex(input_dim):
    inp = layers.Input(shape=(input_dim,), name="tabular_input")
    x = layers.BatchNormalization()(inp)
    x = layers.GaussianNoise(0.01)(x)
    x = dense_block(x, 256, activation="swish", l2_reg=1e-3, dropout_rate=0.30, use_bn=True)
    x = dense_block(x, 128, activation="swish", l2_reg=1e-3, dropout_rate=0.20, use_bn=True)
    x = dense_block(x, 256, activation="swish", l2_reg=1e-3, dropout_rate=0.10, use_bn=True)
    prob = layers.Dense(1, activation="sigmoid", name="prob")(x)
    return tf.keras.Model(inp, prob, name="compact_mlp_nonconvex")



def transfer_weights_best_effort(src_model, dst_model):
    src_weights = {w.name: w.numpy() for w in src_model.weights}
    for w in dst_model.weights:
        if w.name in src_weights and src_weights[w.name].shape == tuple(w.shape):
            w.assign(src_weights[w.name])



def get_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=REDUCE_LR_PATIENCE,
            min_lr=DEEP_MIN_LR,
            verbose=0,
        ),
    ]



def run_schedule(schedule, Xtr, ytr, Xva, yva, class_weight=None, batch_size=32, lr=1e-3, verbose=0):
    input_dim = Xtr.shape[1]
    model = None

    for st in schedule:
        if st["builder"] == "standard":
            new_model = build_compact_mlp_standard(input_dim)
        elif st["builder"] == "convex":
            new_model = build_compact_mlp_convex(input_dim)
        else:
            new_model = build_compact_mlp_nonconvex(input_dim)

        if model is not None:
            transfer_weights_best_effort(model, new_model)

        model = new_model
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=5.0)
        model.compile(
            optimizer=optimizer,
            loss=st["loss"],
            metrics=[tf.keras.metrics.BinaryAccuracy(name="acc"), tf.keras.metrics.AUC(name="auc")],
        )

        model.fit(
            Xtr,
            ytr,
            validation_data=(Xva, yva),
            epochs=st["epochs"],
            batch_size=batch_size,
            verbose=verbose,
            class_weight=class_weight,
            callbacks=get_callbacks(),
        )

    last_builder = schedule[-1]["builder"]
    if last_builder == "convex":
        logits = model.predict(Xva, verbose=0).reshape(-1)
        prob = tf.nn.sigmoid(logits).numpy()
    else:
        prob = model.predict(Xva, verbose=0).reshape(-1)

    prob = np.nan_to_num(prob, nan=0.5, posinf=1.0, neginf=0.0)
    #prob = np.clip(prob, 1e-7, 1.0 - 1e-7)
    return prob



def make_adaptive_schedules(y_train):
    class_weight = compute_class_weight_dict(y_train)

    mlp_cfg = DEEP_FOCAL_CONFIG.get(
        "MLP",
        {"alpha_pos": 0.25, "alpha_neg": 0.85, "gamma_convex": 0.0, "gamma_nonconvex": 2.0},
    )

    standard_bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)

    convex_loss = ConvexFocalLoss(
        gamma=mlp_cfg["gamma_convex"],
        alpha_pos=mlp_cfg["alpha_pos"],
        alpha_neg=mlp_cfg["alpha_neg"],
    )

    nonconvex_focal = tf.keras.losses.BinaryFocalCrossentropy(
        apply_class_balancing=True,
        alpha=mlp_cfg["alpha_pos"],
        gamma=mlp_cfg["gamma_nonconvex"],
        from_logits=False,
    )

    schedule_standard = [
        {"stage": "StandardBCE", "builder": "standard", "loss": standard_bce, "epochs": DEEP_EPOCHS_STANDARD}
    ]
    schedule_convex = [
        {"stage": "ConvexWarmStart", "builder": "convex", "loss": convex_loss, "epochs": DEEP_EPOCHS_CONVEX}
    ]
    schedule_nonconvex = [
        {"stage": "NonConvexFocal", "builder": "nonconvex", "loss": nonconvex_focal, "epochs": DEEP_EPOCHS_NONCONVEX}
    ]
    schedule_multistage = [
        {"stage": "Stage1_ConvexWarmStart", "builder": "convex", "loss": convex_loss, "epochs": DEEP_EPOCHS_STAGE1},
        {"stage": "Stage2_NonConvexFocal", "builder": "nonconvex", "loss": nonconvex_focal, "epochs": DEEP_EPOCHS_STAGE2},
    ]

    return class_weight, schedule_standard, schedule_convex, schedule_nonconvex, schedule_multistage


# =========================================================
# 10) OOF prediction runner
# =========================================================
def generate_oof_predictions(imputer_name, baseline_sampler_name, deep_sampler_name):
    global current_imputer_name
    current_imputer_name = imputer_name

    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    all_models = get_all_models()

    model_oof = {name: np.zeros(len(y_np), dtype=float) for name in all_models.keys()}
    feature_freq = {}

    if RUN_DEEP_MODELS:
        model_oof["MLP_Standard"] = np.zeros(len(y_np), dtype=float)
        model_oof["MLP_Focal_Convex"] = np.zeros(len(y_np), dtype=float)
        model_oof["MLP_Focal_NonConvex"] = np.zeros(len(y_np), dtype=float)
        model_oof["MLP_Focal_Multistage"] = np.zeros(len(y_np), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_np, y_np), start=1):
        print(f"  Fold {fold}/{K}")

        X_train, X_val = X_np[tr_idx], X_np[va_idx]
        y_train, y_val = y_np[tr_idx], y_np[va_idx]

        X_train_fs, X_val_fs, selector, selected_names = preprocess_fold(X_train, X_val, y_train)
        for name in selected_names:
            feature_freq[name] = feature_freq.get(name, 0) + 1

        Xtr_base, ytr_base = apply_sampling(baseline_sampler_name, X_train_fs, y_train, ratio=1)
        for model_name, model in all_models.items():
            probs = get_model_probabilities(model, Xtr_base, ytr_base, X_val_fs)
            model_oof[model_name][va_idx] = probs

        if RUN_DEEP_MODELS:
            Xtr_deep, ytr_deep = apply_sampling(deep_sampler_name, X_train_fs, y_train, ratio=1)

            Xtr_tf = Xtr_deep.astype(np.float32)
            Xva_tf = X_val_fs.astype(np.float32)
            ytr_tf = ytr_deep.astype(np.float32).reshape(-1, 1)
            yva_tf = y_val.astype(np.float32).reshape(-1, 1)

            class_weight, schedule_standard, schedule_convex, schedule_nonconvex, schedule_multistage = make_adaptive_schedules(ytr_deep)

            model_oof["MLP_Standard"][va_idx] = run_schedule(
                schedule_standard, Xtr_tf, ytr_tf, Xva_tf, yva_tf,
                class_weight=class_weight, batch_size=DEEP_BATCH_SIZE, lr=DEEP_LR, verbose=0
            )
            model_oof["MLP_Focal_Convex"][va_idx] = run_schedule(
                schedule_convex, Xtr_tf, ytr_tf, Xva_tf, yva_tf,
                class_weight=class_weight, batch_size=DEEP_BATCH_SIZE, lr=DEEP_LR, verbose=0
            )
            model_oof["MLP_Focal_NonConvex"][va_idx] = run_schedule(
                schedule_nonconvex, Xtr_tf, ytr_tf, Xva_tf, yva_tf,
                class_weight=class_weight, batch_size=DEEP_BATCH_SIZE, lr=DEEP_LR, verbose=0
            )

            p1 = run_schedule(
                schedule_multistage, Xtr_tf, ytr_tf, Xva_tf, yva_tf,
                class_weight=class_weight, batch_size=DEEP_BATCH_SIZE, lr=DEEP_LR, verbose=0
            )
            p2 = run_schedule(
                schedule_multistage, Xtr_tf, ytr_tf, Xva_tf, yva_tf,
                class_weight=class_weight, batch_size=DEEP_BATCH_SIZE, lr=DEEP_LR * 0.8, verbose=0
            )
            model_oof["MLP_Focal_Multistage"][va_idx] = 0.5 * (p1 + p2)

            tf.keras.backend.clear_session()
            gc.collect()

    return model_oof, feature_freq


# =========================================================
# 11) Cross-validated evaluation
# =========================================================
all_summary_rows = []
best_config = None
best_score = None
overall_feature_freq = {}

for imputer_name in IMPUTATION_METHODS:
    for baseline_sampler_name in BASELINE_SAMPLING_METHODS:
        for deep_sampler_name in DEEP_SAMPLING_METHODS:
            print(
                f"\n=== Config: imputer={imputer_name}, baseline_sampler={baseline_sampler_name}, "
                f"deep_sampler={deep_sampler_name}, fs={FEATURE_SELECTION} ==="
            )

            model_oof, feature_freq = generate_oof_predictions(
                imputer_name=imputer_name,
                baseline_sampler_name=baseline_sampler_name,
                deep_sampler_name=deep_sampler_name,
            )

            for k, v in feature_freq.items():
                overall_feature_freq[k] = overall_feature_freq.get(k, 0) + v

            for model_name, probs in model_oof.items():
                if USE_THRESHOLD_TUNING:
                    best_thr, _ = tune_threshold(
                        y_np, probs, metric=THRESHOLD_METRIC, thresholds=THRESHOLD_GRID
                    )
                else:
                    best_thr = DEFAULT_THRESHOLD

                # Correctly use tuned threshold
                m = eval_binary_full(y_np, probs, thr=best_thr)
                weighted_average = compute_weighted_average(m)

                row = {
                    "imputer": imputer_name,
                    "baseline_sampler": baseline_sampler_name,
                    "deep_sampler": deep_sampler_name,
                    "feature_selection": FEATURE_SELECTION,
                    "model": model_name,
                    "accuracy": m["accuracy"],
                    "auc": m["auc"],
                    "specificity": m["specificity"],
                    "threshold": m["threshold"],
                    "weighted_precision": m["weighted_precision"],
                    "weighted_recall": m["weighted_recall"],
                    "weighted_f1": m["weighted_f1"],
                    "weighted_average": weighted_average,
                    "fraud_precision": m["fraud_precision"],
                    "fraud_recall": m["fraud_recall"],
                    "fraud_f1": m["fraud_f1"],
                }
                all_summary_rows.append(row)

                current_score = (
                    m["fraud_recall"],
                    m["auc"],
                    m["fraud_f1"],
                    weighted_average,
                    m["weighted_f1"],
                    m["weighted_precision"],
                    m["specificity"],
                    m["accuracy"],
                )

                if best_config is None or current_score > best_score:
                    best_score = current_score
                    best_config = row.copy()


# =========================================================
# 12) Results summary
# =========================================================
summary = pd.DataFrame(all_summary_rows).sort_values(METRIC_PRIORITY, ascending=[False] * len(METRIC_PRIORITY))

print("\nTop results:")
print(summary.head(40).to_string(index=False))

print("\nBest configuration:")
print(best_config)


# =========================================================
# 13) Best-per-model table
# =========================================================
best_per_model = (
    summary.sort_values(["model"] + METRIC_PRIORITY, ascending=[True] + [False] * len(METRIC_PRIORITY))
    .groupby("model", as_index=False)
    .first()
    .sort_values(METRIC_PRIORITY, ascending=[False] * len(METRIC_PRIORITY))
)

print("\nBest row per model:")
print(best_per_model.to_string(index=False))


# =========================================================
# 14) Family comparison tables
# =========================================================
families = ["RandomForest", "Bagging", "LogisticRegression", "XGBoost", "Stacking", "MLP"]
family_comparison_rows = []

for fam in families:
    if fam == "MLP":
        variant_names = [
            "MLP_Standard",
            "MLP_Focal_Convex",
            "MLP_Focal_NonConvex",
            "MLP_Focal_Multistage",
        ]
    else:
        variant_names = [
            f"{fam}_Standard",
            f"{fam}_Focal_Convex",
            f"{fam}_Focal_NonConvex",
            f"{fam}_Focal_Multistage",
        ]

    fam_df = best_per_model[best_per_model["model"].isin(variant_names)].copy()
    if len(fam_df) == 0:
        continue

    print(f"\n=== {fam} comparison ===")
    print(
        fam_df[
            [
                "model",
                "auc",
                "weighted_precision",
                "weighted_recall",
                "weighted_f1",
                "fraud_precision",
                "fraud_recall",
                "fraud_f1",
                "specificity",
                "accuracy",
                "weighted_average",
            ]
        ].to_string(index=False)
    )

    standard_row = fam_df[fam_df["model"] == variant_names[0]]
    if len(standard_row) > 0:
        standard_row = standard_row.iloc[0]
        for vname in variant_names[1:]:
            vrow = fam_df[fam_df["model"] == vname]
            if len(vrow) == 0:
                continue
            vrow = vrow.iloc[0]
            family_comparison_rows.append(
                {
                    "family": fam,
                    "standard_model": variant_names[0],
                    "variant_model": vname,
                    "delta_auc": vrow["auc"] - standard_row["auc"],
                    "delta_weighted_precision": vrow["weighted_precision"] - standard_row["weighted_precision"],
                    "delta_weighted_recall": vrow["weighted_recall"] - standard_row["weighted_recall"],
                    "delta_weighted_f1": vrow["weighted_f1"] - standard_row["weighted_f1"],
                    "delta_fraud_precision": vrow["fraud_precision"] - standard_row["fraud_precision"],
                    "delta_fraud_recall": vrow["fraud_recall"] - standard_row["fraud_recall"],
                    "delta_fraud_f1": vrow["fraud_f1"] - standard_row["fraud_f1"],
                    "delta_specificity": vrow["specificity"] - standard_row["specificity"],
                    "delta_accuracy": vrow["accuracy"] - standard_row["accuracy"],
                    "delta_weighted_average": vrow["weighted_average"] - standard_row["weighted_average"],
                }
            )

family_comparison_df = pd.DataFrame(family_comparison_rows)

print("\nStandard vs each focal variant:")
print(family_comparison_df.to_string(index=False))


# =========================================================
# 15) ROC curves for top models
# =========================================================
#top_models = best_per_model["model"].head(8).tolist()
#print("\nTop models:", top_models)

#best_imp = best_config["imputer"]
#best_baseline_sampler = best_config["baseline_sampler"]
#best_deep_sampler = best_config["deep_sampler"]

#best_model_oof, _ = generate_oof_predictions(
#    imputer_name=best_imp,
 #   baseline_sampler_name=best_baseline_sampler,
#    deep_sampler_name=best_deep_sampler,
#)

#curves = [(mdl, best_model_oof[mdl]) for mdl in top_models]
#plot_roc_curves(y_np, curves, title="Top Model ROC Curves")


# =========================================================
# 16) Feature frequency report
# =========================================================
#feature_freq_df = pd.DataFrame(
#    [{"feature": k, "count": v} for k, v in overall_feature_freq.items()]
#).sort_values("count", ascending=False)

#print("\nMost frequently selected features:")
#print(feature_freq_df.head(20).to_string(index=False))


# =========================================================
# 17) Export
# =========================================================
summary.to_csv("fraud_pipeline_all_results_4_variants_per_model.csv", index=False)
best_per_model.to_csv("fraud_pipeline_best_per_model_4_variants_per_model.csv", index=False)
family_comparison_df.to_csv("fraud_pipeline_family_variant_deltas.csv", index=False)
#feature_freq_df.to_csv("fraud_pipeline_feature_frequency.csv", index=False)

print("\nSaved:")
print(" - fraud_pipeline_all_results_4_variants_per_model.csv")
print(" - fraud_pipeline_best_per_model_4_variants_per_model.csv")
print(" - fraud_pipeline_family_variant_deltas.csv")
print(" - fraud_pipeline_feature_frequency.csv")



Target column detected: fraud reported
Original dtype: object
Unique sample values: ['N' 'Y']

Target distribution:
fraud reported
0    9043
1    2959
Name: count, dtype: int64

Target proportions:
fraud reported
0    0.7535
1    0.2465
Name: proportion, dtype: float64

=== Config: imputer=knn, baseline_sampler=none, deep_sampler=none, fs=mutual_info ===
  Fold 1/10
  Fold 2/10
  Fold 3/10
  Fold 4/10
  Fold 5/10
  Fold 6/10
  Fold 7/10
  Fold 8/10
  Fold 9/10
  Fold 10/10

Top results:
imputer baseline_sampler deep_sampler feature_selection                               model  accuracy      auc  specificity  threshold  weighted_precision  weighted_recall  weighted_f1  weighted_average  fraud_precision  fraud_recall  fraud_f1
    knn             none         none       mutual_info               Stacking_Focal_Convex  0.392601 0.646473     0.197611       0.49            0.810221         0.392601     0.357632          0.503164         0.287300      0.988510  0.445205
    knn            

In [ ]:
pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 771.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 108.9 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
